In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted successfully")

In [ ]:
# 1. Installing llama-cpp-python with GPU support
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.77 --force-reinstall --no-cache-dir
print("llama-cpp-python installed")

# 2. Installing LangChain and other libraries
print("Installing LangChain and other libraries")
!pip install langchain==0.2.1 langchain-community==0.2.1 langchain-core==0.2.3 PyYAML tqdm
print("All required libraries installed")
print("Installation complete")
import os
os.kill(os.getpid(), 9)

Installing llama-cpp-python with CUDA support
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 MB 213.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 169.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 276.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 312.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 238.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 283.4 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.2.77-cp311-cp311-linux_x86_64.whl size=132734437 sha256=c2982106bd4ce059a6a0d7718b8f5c12e2b25463d110ee77fcb01114732d1fbe
  Stored in directory: /tmp/pip-ephem-wheel-cache-jvha5j4g/wheels/ec/30/e8/495d5a788bab9ea9dafc77c21bd517b

In [ ]:
import json
import os
import time
import logging
import re
from typing import Dict

# LangChain components
from langchain_community.llms import LlamaCpp
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
# from langchain_community.vectorstores import FAISS
# from langchain_huggingface import HuggingFaceEmbeddings
from langchain.docstore.document import Document

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

print("All libraries imported and logging configured")

All libraries imported and logging configured


In [ ]:
# 1. Data and model paths on Google Drive
DRIVE_BASE_PATH = '/content/drive/MyDrive/Competition'

DATA_PATH = os.path.join(DRIVE_BASE_PATH, 'ALQAC_2025/data')
LAW_FILE = os.path.join(DATA_PATH, "alqac25_law.json")
TEST_TASK1_FILE = os.path.join(DATA_PATH, "alqac25_private_test_Task_1.json")
TEST_TASK2_FILE = os.path.join(DATA_PATH, "alqac25_private_test_task2.json")

MODEL_CACHE_DIR = '/content/drive/MyDrive/model_cache'
MODEL_FILENAME = "vinallama-7b-chat_q5_0.gguf"
MODEL_PATH = os.path.join(MODEL_CACHE_DIR, 'models--vilm--vinallama-7b-chat-GGUF/snapshots/5c76606edd7f6c714fba2988990dedecba33c0ff/', MODEL_FILENAME)


# 2. Configuration for LLM (LlamaCpp) in Task 2
LLM_CONFIG = {
    'n_gpu_layers': 35,
    'n_ctx': 4096,
    'temperature': 0.05,
    'max_tokens': 512,
}

# 3. Testing
print(f"Data directory: {DATA_PATH}")
print(f"Law file path: {LAW_FILE}")
print(f"Model file path: {MODEL_PATH}")
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)

Data directory: /content/drive/MyDrive/Competition/ALQAC_2025/data
Law file path: /content/drive/MyDrive/Competition/ALQAC_2025/data/alqac25_law.json
Model file path: /content/drive/MyDrive/model_cache/models--vilm--vinallama-7b-chat-GGUF/snapshots/5c76606edd7f6c714fba2988990dedecba33c0ff/vinallama-7b-chat_q5_0.gguf


In [ ]:
if not os.path.isfile(MODEL_PATH):
    print("Model not found. Download again")
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download(
            repo_id="vilm/vinallama-7b-chat-GGUF",
            filename=MODEL_FILENAME,
            cache_dir=MODEL_CACHE_DIR,
            local_dir=os.path.dirname(MODEL_PATH),
            local_dir_use_symlinks=False
        )
        print("Model downloaded successfully")
    except ImportError:
        print("huggingface_hub is not installed. Run the installation cell again")
    except Exception as e:
        print(f"ERROR: Failed to download model. Reason: {e}")
else:
    print("Model already exists. Skipping download")

Model already exists. Skipping download


In [ ]:
def load_and_process_laws(law_file_path: str):
    """
    Download the law file, process it, and convert it into a list of LangChain documents
    Create a dictionary for quick lookup of legal content
    """
    logging.info(f"Starting to load and process law data from {law_file_path}...")
    with open(law_file_path, 'r', encoding='utf-8') as f:
        law_data = json.load(f)

    all_documents = []
    law_lookup_dict = {}

    for law in law_data:
        law_id = law['id']
        for article in law['articles']:
            article_id = article['id']
            article_text = article['text']

            full_text_content = f"Trích Luật: {law_id}\nĐiều số: {article_id}\nNội dung: {article_text}"
            metadata = {"law_id": law_id, "article_id": article_id}

            doc = Document(page_content=full_text_content, metadata=metadata)
            all_documents.append(doc)

            law_lookup_dict[(law_id, article_id)] = article_text

    logging.info(f"Loaded and processed {len(all_documents)} articles.")
    logging.info(f"Created a lookup dictionary with {len(law_lookup_dict)} entries")

    return all_documents, law_lookup_dict
all_law_documents, law_lookup_dict = load_and_process_laws(LAW_FILE)

In [ ]:
def initialize_llm(model_path: str, llm_config: dict) -> LlamaCpp:
    """Initialize and return an instance of the LlamaCpp LLM from LangChain"""
    logging.info(f"Initializing LLM from path: {model_path}")
    print(model_path)
    try:
        llm = LlamaCpp(
            model_path=model_path,
            n_gpu_layers=llm_config.get('n_gpu_layers', -1),
            n_batch=512,
            n_ctx=llm_config.get('n_ctx', 4096),
            f16_kv=True,
            temperature=llm_config.get('temperature', 0.1),
            max_tokens=llm_config.get('max_tokens', 150),
            verbose=False,
        )
        logging.info("LangChain LlamaCpp initialized successfully")
        return llm
    except Exception as e:
        logging.error(f"Failed to load LLM model: {e}")
        return None

def create_qa_chains(llm: LlamaCpp) -> Dict[str, LLMChain]:
    """Create different LLMChains for each question type with detailed prompts"""
    # True/ False Question
    true_false_template = """
    <|im_start|>system
    Bạn là một trợ lý pháp lý AI chuyên nghiệp. Bạn được giao nhiệm vụ xác định tính ĐÚNG/SAI của một phát biểu dựa trên nội dung trích từ một điều luật hoặc điều khoản pháp lý cụ thể
    - Không thêm kiến thức ngoài văn bản
    - Chỉ trả lời một từ duy nhất: "Đúng" hoặc "Sai".<|im_end|>

    <|im_start|>user
    ### Nhiệm vụ của bạn:
    1. Đọc kỹ phần **Ngữ cảnh** bên dưới (nội dung của điều luật hoặc điều khoản pháp lý)
    2. Đọc kỹ phần **Phát biểu**
    3. Dựa vào thông tin trong **Ngữ cảnh**, xác định xem **Phát biểu** là **Đúng** hay **Sai**
    4. **Chỉ** trả lời bằng một từ duy nhất: **"Đúng"** hoặc **"Sai"** (không giải thích thêm)

    **Ví dụ**:

    Ngữ cảnh:
    > Người nghiện ma túy từ đủ 18 tuổi trở lên bị áp dụng biện pháp xử lý hành chính đưa vào cơ sở cai nghiện bắt buộc nếu sử dụng trái phép chất ma túy trong thời gian cai nghiện ma túy tự nguyện.

    Phát biểu:
    > Người nghiện ma túy từ đủ 18 tuổi trở lên bị đưa vào cơ sở cai nghiện bắt buộc nếu bị phát hiện sử dụng trái phép chất ma túy trong thời gian đang cai nghiện tự nguyện.

    Trả lời: Đúng

    ---

    **Bây giờ, hãy áp dụng quy trình trên với dữ liệu sau:**

    --- Ngữ cảnh ---
    {context}
    --- Hết Ngữ cảnh ---

    **Phát biểu:** "{statement}"

    **Chỉ trả lời bằng một từ duy nhất: "Đúng" hoặc "Sai"**<|im_end|>

    <|im_start|>assistant
    """

    true_false_prompt = PromptTemplate(template=true_false_template, input_variables=['context', 'statement'])

    # Multiple Choice
    multiple_choice_template = """
    <|im_start|>system
    Bạn là một trợ lý pháp lý AI xuất sắc. Nhiệm vụ của bạn là đọc và hiểu nội dung pháp luật được cung cấp, sau đó trả lời một câu hỏi trắc nghiệm bằng cách chọn ra một phương án đúng nhất. Câu trả lời của bạn phải là một chữ cái duy nhất đại diện cho lựa chọn đúng (ví dụ: A, B, C hoặc D). Tuyệt đối không thêm giải thích hay từ ngữ dư thừa.<|im_end|>

    <|im_start|>user
    ### Hướng dẫn thực hiện:
    1. Đọc kỹ phần "Ngữ cảnh pháp lý" – đây là điều luật được trích dẫn nguyên văn
    2. Đọc kỹ câu hỏi và 4 phương án lựa chọn A, B, C, D
    3. Dựa *duy nhất vào nội dung của điều luật* (không dùng kiến thức ngoài) để chọn đáp án đúng nhất
    4. Trả lời bằng *một ký tự duy nhất*: "A", "B", "C", hoặc "D". *Không giải thích*, không viết thêm bất kỳ từ nào

    ---

    *Ví dụ:*

    Ngữ cảnh pháp lý:
    Điều 29. Đơn phương chấm dứt hợp đồng làm việc của đơn vị sự nghiệp công lập
    Đơn vị sự nghiệp công lập được đơn phương chấm dứt hợp đồng làm việc với viên chức trong trường hợp:
    a) Viên chức có 02 năm liên tiếp bị phân loại đánh giá chất lượng ở mức không hoàn thành nhiệm vụ;
    (Các điểm khác không liên quan đã được lược bỏ)

    Câu hỏi:
    Viên chức bị đơn vị sự nghiệp đơn phương chấm dứt hợp đồng trong trường hợp nào?

    Các lựa chọn:
    A. Viên chức có 02 năm liên tiếp bị phân loại đánh giá ở mức độ không hoàn thành nhiệm vụ
    B. Viên chức ốm đau hoặc bị tai nạn, đang điều trị bệnh nghề nghiệp theo quyết định của cơ sở chữa bệnh
    C. Viên chức đang nghỉ hàng năm, nghỉ về việc riêng và những trường hợp nghỉ khác được người đứng đầu đơn vị sự nghiệp công lập cho phép
    D. Viên chức nữ đang trong thời gian có thai, nghỉ thai sản, nuôi con dưới 36 tháng tuổi

    Trả lời: A

    ---

    Bây giờ, hãy áp dụng đúng quy trình trên với câu hỏi sau:

    --- Ngữ cảnh pháp lý ---
    {context}
    --- Hết Ngữ cảnh ---

    Câu hỏi:
    "{question}"

    Các lựa chọn:
    {choices}

    Chỉ trả lời bằng *một chữ cái duy nhất*: A, B, C hoặc D. Không viết gì thêm nữa

    Đáp án:<|im_end|>

    <|im_start|>assistant
    """

    multiple_choice_prompt = PromptTemplate(template=multiple_choice_template, input_variables=["context", "question", "choices"])

    # Free Text Question
    free_text_template = """<|im_start|>system
    Bạn là một trợ lý pháp lý AI. Dựa vào "Ngữ cảnh" được cung cấp, hãy trả lời câu hỏi một cách trực tiếp, ngắn gọn và chính xác nhất có thể. Chỉ đưa ra câu trả lời cho câu hỏi, không thêm thông tin nền, lời chào hay giải thích dài dòng.<|im_end|>
    <|im_start|>user
    ### Hướng dẫn:
    1. Đọc kỹ phần "Ngữ cảnh pháp lý" – đây là một điều luật cụ thể có liên quan đến câu hỏi
    2. Trả lời câu hỏi *ngắn gọn nhất có thể*, thường là một cụm từ, con số, mốc thời gian, v.v
    3. Tuyệt đối *không đưa thêm lý do, ví dụ, hoặc giải thích gì thêm*
    4. *Chỉ sử dụng thông tin có trong ngữ cảnh* – không suy đoán hay dùng kiến thức ngoài
    ---
    Ví dụ minh hoạ:
    Ngữ cảnh pháp lý:
    Trong thời hạn 10 ngày kể từ ngày nhận được đơn khởi kiện, Trung tâm trọng tài phải gửi cho bị đơn bản sao đơn khởi kiện của nguyên đơn và những tài liệu kèm theo, trừ trường hợp các bên có thỏa thuận khác hoặc quy tắc tố tụng của Trung tâm trọng tài có quy định khác


    Câu hỏi:
    Trong trường hợp các bên không có thỏa thuận khác hoặc quy tắc tố tụng của trung tâm trọng tài không có quy định khác, Trung tâm trọng tài phải gửi cho bị đơn bản sao đơn khởi kiện của nguyên đơn và những tài liệu theo quy định trong thời hạn bao lâu kể từ ngày nhận được đơn khởi kiện?


    Câu trả lời: *10 ngày*
    ---

    Bây giờ, hãy áp dụng đúng quy trình trên với câu hỏi sau:

    Ngữ cảnh:
    {context}

    Câu hỏi:
    "{question}"

    Câu trả lời ngắn gọn:<|im_end|>

    <|im_start|>assistant
    """
    free_text_prompt = PromptTemplate(template=free_text_template, input_variables=["context", "question"])

    logging.info("Created prompt templates for 3 question types")
    return {
        "Đúng/Sai": LLMChain(prompt=true_false_prompt, llm=llm),
        "Trắc nghiệm": LLMChain(prompt=multiple_choice_prompt, llm=llm),
        "Tự luận": LLMChain(prompt=free_text_prompt, llm=llm)
    }

def post_process_answer(raw_answer: str, question_type: str) -> str:
    """Post-process the LLM’s answer"""
    answer = raw_answer.strip()

    if question_type == "Đúng/Sai":
        ans_lower = answer.lower()
        if "đúng" in ans_lower: return "Đúng"
        if "sai" in ans_lower: return "Sai"
        return "Sai" # Mặc định

    elif question_type == "Trắc nghiệm":
        match = re.search(r'\b[A-D]\b', answer, re.IGNORECASE)
        if match: return match.group(0).upper()
        return "A" # Mặc định

    elif question_type == "Tự luận":
        answer = re.sub(r'^(câu trả lời là|dựa trên ngữ cảnh,|trả lời:)\s*', '', answer, flags=re.IGNORECASE)
        if answer.endswith('.'): answer = answer[:-1]
        return answer.strip()

    return answer

print("Helper functions for Task 2 are ready")

In [ ]:
def run_task_2(test_file_path: str, model_path: str, llm_config: dict, lookup_dict: dict):
    """
    Coordinate the execution of Task2 using the above helper functions
    """
    logging.info("Starting Task 2: Legal Question Answering")
    print(model_path)
    # 1. Iniialize LLM
    llm = initialize_llm(model_path, llm_config)
    if not llm:
        logging.error("ERROR: LLM initialization failure")
        return []

    # 2. Create QA chains
    qa_chains = create_qa_chains(llm)

    # 3. Load test dataset
    logging.info(f"Loading test data from {test_file_path}")
    with open(test_file_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    logging.info(f"Found {len(test_data)} questions for Task 2")

    # 4. Process questions
    submission_results = []
    for i, item in enumerate(test_data):
        question_id = item['question_id']
        question_type = item['question_type']
        question_text = item['text']
        relevant_articles_info = item['relevant_articles']

        logging.info(f"Processing question {i+1}/{len(test_data)}: {question_id} (Type: {question_type})")

        context_parts = []
        for ref in relevant_articles_info:
          law_id = ref['law_id']
          article_id = ref['article_id']
          article_text = lookup_dict.get((law_id, article_id))

        context = "\n\n".join(context_parts)
        chain = qa_chains.get(question_type)
        if not chain:
            logging.warning(f"No chain found for question type '{question_type}'. Skipping.")
            continue

        # Prepare input appropriate for each prompt
        input_data = {"context": context}
        if question_type == "Đúng/Sai":
            input_data["statement"] = question_text
        else: # Trắc nghiệm và Tự luận
            input_data["question"] = question_text

        if question_type == "Trắc nghiệm":
            input_data["choices"] = "\n".join([f"{k}: {v}" for k, v in item['choices'].items()])

        # Call chain and post-process the output
        try:
            raw_answer = chain.invoke(input_data)['text']
            final_answer = post_process_answer(raw_answer, question_type)
            logging.info(f"Raw answer: '{raw_answer.strip()}' -> Final answer: '{final_answer}'")

            submission_results.append({"question_id": question_id, "answer": final_answer})
        except Exception as e:
            logging.error(f"Error processing question {question_id}: {e}")
            # Default case
            submission_results.append({"question_id": question_id, "answer": ""})

    # 5. Save the result
    output_filename = "submission_task2.json"
    logging.info(f"Saving Task 2 results to {output_filename}...")
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(submission_results, f, indent=4, ensure_ascii=False)

    logging.info"Task 2 Completed ")
    return submission_results

In [ ]:
task2_results = run_task_2(
        test_file_path=TEST_TASK2_FILE,
        model_path=MODEL_PATH,
        llm_config=LLM_CONFIG,
        lookup_dict=law_lookup_dict
    )
print("Task 2 submission result")
if task2_results:
    print(json.dumps(task2_results[0], indent=2, ensure_ascii=False))

logging.info("Pipeline completed successfully")


/content/drive/MyDrive/model_cache/models--vilm--vinallama-7b-chat-GGUF/snapshots/5c76606edd7f6c714fba2988990dedecba33c0ff/vinallama-7b-chat_q5_0.gguf
/content/drive/MyDrive/model_cache/models--vilm--vinallama-7b-chat-GGUF/snapshots/5c76606edd7f6c714fba2988990dedecba33c0ff/vinallama-7b-chat_q5_0.gguf
Task 2 submission result
{
  "question_id": "private_test_alquac25_1",
  "answer": "Sai"
}
